# 19. PM10 시계열 데이터 입문

이번 장부터는 시간 순서가 있는 데이터를 다룹니다.

PM10 데이터는 시간별 미세먼지 농도 데이터입니다.

이번 장의 목표:

```text
CSV 읽기
-> 날짜 변환
-> 지역 선택
-> 결측치 확인
-> 시간 흐름 시각화
```

## 1. 라이브러리 불러오기

이번 장에서는 표 데이터를 읽고 시간 흐름을 그리기 위해 `pandas`와 `matplotlib`을 사용합니다.

In [ ]:
from pathlib import Path

# numpy는 숫자 계산을 위한 라이브러리입니다.
import numpy as np

# pandas는 CSV 파일을 읽고 표 형태 데이터를 다루는 라이브러리입니다.
import pandas as pd

# matplotlib.pyplot은 그래프를 그리는 라이브러리입니다.
import matplotlib.pyplot as plt

## 2. CSV 파일 읽기

이 파일은 UTF-8이 아니라 `cp949` 인코딩으로 읽어야 합니다.

한국어 CSV 파일에서는 인코딩 문제가 자주 생깁니다.

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\seoul_pm10.csv")

print("파일 존재 여부:", DATA_PATH.exists())

# read_csv()는 CSV 파일을 DataFrame으로 읽습니다.
# encoding="cp949"는 한국어 Windows 계열 인코딩으로 읽겠다는 뜻입니다.
df = pd.read_csv(DATA_PATH, encoding="cp949")

print("데이터 shape:", df.shape)
df.head()

## 3. 컬럼과 데이터 타입 확인

시계열 데이터에서는 날짜 컬럼의 타입이 중요합니다.

처음 읽으면 날짜가 문자열(object)로 들어오는 경우가 많습니다.

In [ ]:
# columns는 컬럼 이름 목록입니다.
print("컬럼 목록:", list(df.columns))

# dtypes는 각 컬럼의 데이터 타입을 보여줍니다.
print("\n데이터 타입:")
print(df.dtypes)

# isna().sum()은 컬럼별 결측치 개수를 셉니다.
print("\n결측치 개수:")
print(df.isna().sum())

## 4. 날짜 컬럼을 datetime으로 변환

문자열 날짜는 시간 계산과 정렬에 불편합니다.

`pd.to_datetime()`을 사용하면 날짜/시간 타입으로 바꿀 수 있습니다.

In [ ]:
# pd.to_datetime()은 문자열 형태의 날짜를 pandas datetime 타입으로 변환합니다.
# errors="coerce"는 변환이 안 되는 값이 있으면 NaT로 바꾸라는 뜻입니다.
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(df.dtypes)
print("날짜 최솟값:", df["date"].min())
print("날짜 최댓값:", df["date"].max())

## 5. 지역 목록 확인

데이터에는 여러 서울 지역구가 들어 있습니다.

처음에는 한 지역만 골라서 시계열 흐름을 단순하게 봅니다.

In [ ]:
# unique()는 중복을 제거한 고유값 목록을 반환합니다.
areas = sorted(df["area"].unique())

print("지역 개수:", len(areas))
print("지역 목록:")
for area in areas:
    print("-", area)

## 6. 한 지역 선택하기

이번 장에서는 `강남구`를 선택합니다.

한 지역만 선택하면 시간별 PM10 흐름을 더 쉽게 볼 수 있습니다.

In [ ]:
TARGET_AREA = "강남구"

# area 컬럼이 강남구인 행만 선택합니다.
area_df = df[df["area"] == TARGET_AREA].copy()

# sort_values()는 날짜 순서대로 데이터를 정렬합니다.
area_df = area_df.sort_values("date")

print("선택 지역:", TARGET_AREA)
print("선택 데이터 shape:", area_df.shape)
area_df.head()

## 7. 시간 순서 확인

시계열 데이터에서는 정렬이 중요합니다.

날짜가 뒤섞인 상태로 모델에 넣으면 시간 흐름을 배울 수 없습니다.

In [ ]:
print("처음 5개 시간:")
print(area_df["date"].head())

print("\n마지막 5개 시간:")
print(area_df["date"].tail())

# diff()는 이전 행과 현재 행의 차이를 계산합니다.
# 시간 간격이 일정한지 볼 때 사용할 수 있습니다.
time_diff = area_df["date"].diff()

print("\n시간 간격 상위 값:")
print(time_diff.value_counts().head())

## 8. 결측치 처리

실제 센서 데이터에는 빈 값이 있을 수 있습니다.

이번 장에서는 먼저 결측치 개수를 확인하고, 간단하게 앞 시간 값으로 채워 봅니다.

In [ ]:
print("결측치 개수:")
print(area_df[["pm10", "pm2.5"]].isna().sum())

# ffill()은 forward fill의 줄임말입니다.
# 바로 앞 시간의 값으로 결측치를 채웁니다.
area_df["pm10_filled"] = area_df["pm10"].ffill()
area_df["pm25_filled"] = area_df["pm2.5"].ffill()

print("\n채운 뒤 결측치 개수:")
print(area_df[["pm10_filled", "pm25_filled"]].isna().sum())

## 9. PM10 시간 흐름 그리기

이제 강남구의 PM10 흐름을 그래프로 봅니다.

시계열에서는 그래프를 먼저 보는 습관이 중요합니다.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(area_df["date"], area_df["pm10_filled"], linewidth=1)
plt.title("Gangnam PM10 Time Series")
plt.xlabel("Date")
plt.ylabel("PM10")
plt.tight_layout()
plt.show()

## 10. 일부 기간만 확대해서 보기

전체 1년을 한 번에 보면 세부 패턴이 잘 안 보일 수 있습니다.

처음 7일만 확대해서 봅니다.

In [ ]:
# 첫 날짜를 기준으로 7일 뒤까지 선택합니다.
start_date = area_df["date"].min()
end_date = start_date + pd.Timedelta(days=7)

week_df = area_df[(area_df["date"] >= start_date) & (area_df["date"] < end_date)]

plt.figure(figsize=(12, 4))
plt.plot(week_df["date"], week_df["pm10_filled"], marker="o", linewidth=1)
plt.title("First 7 Days PM10")
plt.xlabel("Date")
plt.ylabel("PM10")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 11. 이동평균 보기

미세먼지 값은 시간마다 흔들릴 수 있습니다.

이동평균은 가까운 시간들의 평균을 내서 큰 흐름을 보기 쉽게 합니다.

In [ ]:
# rolling(window=24)는 현재 시점 주변의 최근 24개 값을 묶습니다.
# mean()은 그 묶음의 평균을 계산합니다.
# 시간별 데이터에서 24개는 대략 하루 평균으로 볼 수 있습니다.
area_df["pm10_24h_mean"] = area_df["pm10_filled"].rolling(window=24).mean()

plt.figure(figsize=(12, 4))
plt.plot(area_df["date"], area_df["pm10_filled"], alpha=0.35, label="PM10 hourly")
plt.plot(area_df["date"], area_df["pm10_24h_mean"], linewidth=2, label="24-hour moving average")
plt.title("PM10 and 24-hour Moving Average")
plt.xlabel("Date")
plt.ylabel("PM10")
plt.legend()
plt.tight_layout()
plt.show()

## 12. 기본 통계 확인

PM10 값의 최솟값, 최댓값, 평균 등을 확인합니다.

In [ ]:
# describe()는 숫자 컬럼의 기본 통계를 보여줍니다.
area_df[["pm10_filled", "pm25_filled"]].describe()

## 13. 다음 장을 위한 단일 시계열 만들기

다음 장에서는 LSTM 입력을 만들기 위해 PM10 값만 따로 사용할 예정입니다.

딥러닝 모델에는 보통 pandas Series가 아니라 numpy 배열로 넣습니다.

In [ ]:
# values는 pandas Series를 numpy 배열로 꺼냅니다.
pm10_values = area_df["pm10_filled"].values

print("pm10_values shape:", pm10_values.shape)
print("앞 10개 값:")
print(pm10_values[:10])

## 14. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. PM10 데이터는 시간 순서가 있는 시계열 데이터다.
2. 한국어 CSV는 인코딩을 맞춰 읽어야 할 수 있다.
3. 날짜 컬럼은 datetime으로 변환해야 한다.
4. 여러 지역이 있으면 처음에는 한 지역만 골라 단순하게 시작할 수 있다.
5. 시계열에서는 결측치, 정렬, 시간 흐름 시각화가 중요하다.
```

## 과제

1. `TARGET_AREA`를 다른 지역으로 바꿔서 흐름을 비교해보세요.
2. PM10과 PM2.5의 차이를 그래프로 비교해보세요.
3. 이동평균 window를 24가 아니라 6, 72로 바꾸면 그래프가 어떻게 달라질지 확인해보세요.
4. 시계열 데이터에서 무작위 train/test split을 조심해야 하는 이유를 적어보세요.